# Spartan Career Compass - Auto-setup Notebook

**CMPE 259 Term Project - Hriday Ampavatina (Team #5)**

This notebook walks an evaluator from "fresh environment" to "first answer"
end to end. It runs the same code as the Streamlit app (`app.py`),
exercising all five tools, all three prompting modes, and the prompt
cache against the deployed mistral 7B and llama2 13B models.

## Environments

The notebook auto-detects whether it is running in Colab or in a local
Jupyter / VS Code kernel and only fires the Colab-specific install cells
when needed. For a local run, activate the project venv first
(`Code/.venv`) and select it as the kernel before running.

## Section map

1. Environment detection
2. Install dependencies (Colab) or sanity-check the venv (local)
3. Initialize the SQLite database and ingest guides + scraped data
4. Install and start Ollama, pull mistral:7b and llama2:13b
5. Tool sanity checks (no LLM calls)
6. Single-question agent run (simple mode)
7. Three prompting modes side by side
8. Prompt-cache benchmark
9. Security testing (5 prompt-injection attacks per model)
10. Model comparison (mistral 7B vs llama2 13B, simple mode)
11. Pointers to the full Evaluation.ipynb

## 1. Environment detection

In [ ]:
import os, sys, platform
IS_COLAB = 'google.colab' in sys.modules
print('Python    :', sys.version.split()[0])
print('Platform  :', platform.platform())
print('In Colab  :', IS_COLAB)

## 2. Install dependencies

In Colab this clones the repo and installs `requirements.txt`. Locally
the venv should already have everything; we just verify imports.

In [ ]:
if IS_COLAB:
    if not os.path.exists('Code'):
        !git clone https://github.com/hriday1231/CMPE-259-Spartan-Career-Compass.git Code
    %cd Code
    !pip install -q -r requirements.txt
else:
    # local venv path: notebook should be opened from Code/
    if os.path.basename(os.getcwd()) != 'Code' and os.path.isdir('Code'):
        os.chdir('Code')
    print('Working dir:', os.getcwd())
    # verify imports
    import langchain, langchain_ollama, streamlit, requests, bs4, pdfplumber
    print('Dependencies OK')

## 3. Initialize the database and ingest data

The database is a single SQLite file at `data/career_compass.db`. SQLite
ships with the Python standard library, no server install needed.

Set up `.env` with `ADZUNA_APP_ID`, `ADZUNA_APP_KEY`, and (optionally)
`BRAVE_API_KEY` before running this cell - copy `.env.example` to `.env`
and fill in the values.

In [ ]:
import os
if not os.path.exists('.env'):
    print('WARNING: .env not found. Copy .env.example to .env and fill in keys before continuing.')
    if os.path.exists('.env.example'):
        with open('.env.example') as f:
            print(f.read())

!python scripts/init_db.py
!python scripts/load_guides.py
!python scripts/scrape_all.py

## 4. Install and start Ollama, pull both models

On Colab the install + pulls take a few minutes the first time. Locally,
if Ollama is already running and the models are pulled, this cell just
confirms availability.

In [ ]:
import subprocess, time, requests as _r

def _ollama_up(base='http://localhost:11434'):
    try:
        return _r.get(f'{base}/api/tags', timeout=2).status_code == 200
    except Exception:
        return False

if IS_COLAB:
    !curl -fsSL https://ollama.com/install.sh | sh
    if not _ollama_up():
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(4)

assert _ollama_up(), 'Ollama is not reachable on localhost:11434 - start `ollama serve` first'

!ollama pull mistral:7b
!ollama pull llama2:13b

## 5. Tool sanity checks (no LLM calls)

Exercise the five tools directly so any DB or API issue surfaces before
the LLM is involved.

In [ ]:
import sys
sys.path.insert(0, '.')
from src.tools import query_events_tool, query_staff_tool, search_guides_tool, search_jobs_tool, web_search_tool

print('--- events ---')
print(query_events_tool.invoke({'days_ahead': 14, 'limit': 5}))
print('\n--- staff ---')
print(query_staff_tool.invoke({'college': 'Engineering'}))
print('\n--- guides ---')
print(search_guides_tool.invoke({'query': 'resume STAR method', 'max_chunks': 3}))
print('\n--- jobs ---')
print(search_jobs_tool.invoke({'keyword': 'data science intern', 'location': 'California', 'limit': 3}))
print('\n--- web ---')
print(web_search_tool.invoke({'query': 'Adobe company overview', 'count': 3}))

## 6. Single-question agent run (simple mode)

In [ ]:
from src.agent import run_agent
print(run_agent('What career events are happening this week?', mode='simple'))

## 7. Three prompting modes side by side

Same question through simple, chain, and reflect modes on mistral 7B.

In [ ]:
q = 'Create a 2-week job search plan.'
for mode in ['simple', 'chain', 'reflect']:
    print(f'\n=== {mode.upper()} ===')
    print(run_agent(q, model_name='mistral:7b', mode=mode, use_cache=False))

## 8. Prompt-cache benchmark (mistral 7B)

Cold = `use_cache=False` (the LLM is invoked); Hit = the second identical
call with `use_cache=True` (returns from SQLite without invoking Ollama).

In [ ]:
!python scripts/benchmark_cache.py --model mistral:7b --mode simple --clear-cache

## 9. Security testing - 5 prompt-injection attacks per model

Five attacks against each model, graded by the regex rubric in
`scripts/security_tests.py`. Reproduces the security findings in
`Evaluation.ipynb`.

In [ ]:
!python scripts/security_tests.py --models mistral:7b,llama2:13b
import json
for r in json.load(open('results/security_results.json')):
    print(f"{r['model']:<14} attack {r['attack_id']} {r['attack_name'][:40]:<42} -> {r['verdict']}")

## 10. Model comparison - mistral 7B vs llama2 13B (simple mode)

Runs the 20-query simple-mode comparison via `run_eval.py`. The full
20-query x 2-model x 3-mode sweep lives in `scripts/compare_models.py`
but takes much longer; this cell sticks to the report-grade subset.

In [ ]:
!python scripts/run_eval.py --parts compare,modes
import json, statistics
rows = json.load(open('results/compare_simple.json'))
by = {}
for r in rows:
    by.setdefault(r['model'], []).append(r)
print(f"{'model':<14}{'avg_latency_ms':>16}{'avg_chars':>12}{'placeholders_total':>22}{'stray_urls_total':>20}")
for m, rs in by.items():
    avg_lat = int(statistics.mean(r['latency_ms'] for r in rs))
    avg_ch = int(statistics.mean(r['answer_chars'] for r in rs))
    ph = sum(r['placeholders'] for r in rs)
    stray = sum(r['urls_not_in_context'] for r in rs)
    print(f"{m:<14}{avg_lat:>16}{avg_ch:>12}{ph:>22}{stray:>20}")

## 11. Pointers

For the full evaluation (security + cache + model + modes, with written
assessments), open `Evaluation.ipynb` in Jupyter or VS Code with the
project venv kernel selected, then Run All to refresh outputs from the
result JSONs.

To use the agent interactively, run:

\\ash
streamlit run app.py
\
and pick a model / prompting mode / cache toggle in the sidebar.